In [ ]:
import requests
import json
import uuid
from IPython.display import display_javascript, display_html, display
import pandas as pd

response = requests.get("http://api.openweathermap.org/data/2.5/forecast?id=5780993&APPID=e0c55e00bf021f6142de334823046e9e&units=imperial")
weather = response.content.decode("utf-8")
weatherDict = json.loads(weather)

In [ ]:
print(json.dumps(weatherDict, indent=2))

In [ ]:
# Method for rendering collapsible JSON from: http://stackoverflow.com/questions/18873066/pretty-json-formatting-in-ipython-notebook

class RenderJSON(object):
    def __init__(self, json_data):
        if isinstance(json_data, dict):
            self.json_str = json.dumps(json_data)
        else:
            self.json_str = json
        self.uuid = str(uuid.uuid4())

    def _ipython_display_(self):
        display_html('<div id="{}" style="height: 600px; width:100%;"></div>'.format(self.uuid),
        raw=True)
        
        display_javascript("""
        require(["https://rawgit.com/caldwell/renderjson/master/renderjson.js"], function() {
        document.getElementById('%s').appendChild(renderjson(%s))
        });
        """ % (self.uuid, self.json_str), raw=True)
        
RenderJSON(weatherDict)

In [ ]:
#pdWeather = pd.read_json(weatherDict)
#pdWeather

#pd.DataFrame(weatherDict["data"], columns=[x["label"] for x in weatherDict["fields"]])

timeList = []
maxTempList = []
minTempList = []
pressureList = []
humidityList = []
tempList = []
wind_speedList = []
windDegList = []
rainList = []
rainListML = []

for weatherEntry in weatherDict["list"]:
    timeList.append(weatherEntry["dt_txt"])
    mainWeather = weatherEntry["main"]
    maxTempList.append(mainWeather["temp_max"])
    minTempList.append(mainWeather["temp_min"])
    pressureList.append(mainWeather["pressure"])
    humidityList.append(mainWeather["humidity"])
    tempList.append(mainWeather["temp"])
    windWeather = weatherEntry["wind"]
    wind_speedList.append(windWeather["speed"])
    windDegList.append(windWeather["deg"])
    if "3h" in weatherEntry["rain"]:
        rainList.append(1)
        rainListML.append(weatherEntry["rain"]["3h"])
    else:
        rainList.append(0)
        rainListML.append(0)
        
    

In [ ]:
rainListML

In [ ]:
data = [('DateTime', timeList),
         ('MaxTemp', maxTempList),
         ('MinTemp', minTempList),
         ('Pressure', pressureList),
         ('Humidity', humidityList),
         ('Temperature', tempList),
         ('WindSpeed', wind_speedList), 
         ('WindDeg', windDegList),
         ('Rain', rainList),
         ('Rain (ml)', rainListML)
         ]

weatherForecast = pd.DataFrame.from_items(data)
weatherForecast["DateTime"] = weatherForecast["DateTime"].apply(lambda x: str(x)[:10])

finalForecast = weatherForecast.groupby("DateTime").mean()

finalForecast["MinTemp"] = weatherForecast.groupby("DateTime").min()["MinTemp"]
finalForecast["MaxTemp"] = weatherForecast.groupby("DateTime").max()["MaxTemp"]
finalForecast["MaxPressure"] = weatherForecast.groupby("DateTime").max()["Pressure"]
finalForecast["MinPressure"] = weatherForecast.groupby("DateTime").min()["Pressure"]
finalForecast["MaxWindSpeed"] = weatherForecast.groupby("DateTime").max()["WindSpeed"]
finalForecast["MinWindSpeed"] = weatherForecast.groupby("DateTime").min()["WindSpeed"]

finalForecast

In [ ]:
finalForecast.info()

In [ ]:
finalForecast.describe()